# 4 · Règle de Bayes — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch04_bayes/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « pourquoi ? » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch04_bayes/03_notebook.ipynb`, créée par `python tools/start_chapter.py 4`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 4.1, 4.2, 4.4, 4.5, 4.6, 4.8 | vérifier tes exercices papier ✏️ | ✏️ | ★ à ★★ | |
| A | 4.12 | Combien de lancers pour démasquer la pièce truquée ? | 🔮 | ★ | 10 |
|  | 4.13 | L'estimation fréquentiste : la moyenne courante des faces | 🔬 | ★ | 15 |
|  | 4.14 | evidence et bayes_posterior | 🔨 | ★★ | 20 |
|  | 4.15 | Bayes chez les manchots : l'espèce sachant l'île | 📦 | ★★ | 20 |
| B | 4.16 | La boucle posterior-prior : update_discrete | 🔨 | ★★ | 30 |
|  | 4.17 | Reproduire les trente lancers de la figure 4.24 | 🎨 | ★★ | 25 |
|  | 4.18 | Le posterior qui s'évanouit : underflow | 🐛 | ★★ | 20 |
|  | 4.19 | La grille biais × proportion de faces | 🔬 | ★★ | 30 |
|  | 4.20 | Envoyer des sondes jusqu'à la décision | 🔬 | ★★ | 25 |
| C | 4.21 | Un prior trompeur centré sur 0,8 | 🔮 | ★★ | 20 |
|  | 4.22 | Le posterior continu : vérifier avec scipy.stats.beta | 📦 | ★★ | 20 |
|  | 4.23 | Refactoriser : du copier-coller à une fonction testée | 🛠️ | ★★ | 20 |
| D | 4.24 | coin_bias_posterior : 500 hypothèses en log-probabilités | 🔨 | ★★★ | 35 |
|  | 4.25 | Intervalle de crédibilité contre intervalle bootstrap | 🔨 | ★★★ | 35 |
|  | 4.26 | Le détective de pièces : vingt pièces, le moins de lancers possible | 🏆 | ★★★ | 60 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="4")  # your own library

## Objectifs et rappel express

- Calculer un posterior avec la règle de Bayes, pour deux puis pour plusieurs hypothèses.
- Mettre à jour une croyance observation après observation (la boucle posterior → prior), sans underflow.
- Relier la règle de Bayes à la precision d'un test et à la prévalence.
- Résumer un posterior par son MAP et un intervalle de crédibilité, et le comparer au bootstrap.

**Rappel express.** $P(H \mid O) = \frac{P(O \mid H)\,P(H)}{P(O)}$ avec $P(O) = \sum_i P(O \mid H_i)\,P(H_i)$ ; le posterior d'une observation est le prior de la suivante ; en log : $\log P(\theta \mid \text{données}) = \log P(\theta) + h\log\theta + t\log(1 - \theta) + \text{cste}$, puis on retranche le maximum avant l'exponentielle ; prior uniforme ⇒ posterior $\mathrm{Beta}(h + 1, t + 1)$. En Python : `np.cumsum`, `np.log`, `scipy.special.logsumexp`, `scipy.stats.beta`.

## Partie 0 · Vérifier tes exercices papier (✏️ 4.1, 4.2, 4.4, 4.5, 4.6 et 4.8)

Fais d'abord les exercices ✏️ de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée (par exemple `42`, `0.125`, `[1, 2, 3]` ou `True`), pas l'expression Python, sinon tu ne vérifies rien. Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les exercices ∂ 4.3 et 4.7 se corrigent avec `05_solutions.md`.

**Ex 4.1 — Une face : la pièce est-elle équilibrée ?**

In [ ]:
# Ex 4.1: the values you found on paper
answer_4_1a = ...  # a) the prior P(fair)
answer_4_1b = ...  # b) [P(heads | fair), P(heads | rigged)], a list
answer_4_1c = ...  # c) the evidence P(heads), 3 decimals
answer_4_1d = ...  # d) P(fair | heads), 2 decimals
answer_4_1e = ...  # e) P(rigged | heads), 2 decimals
answer_4_1f = ...  # f) the expected number of heads in 800 runs
answer_4_1g = ...  # g) how many of these heads come from the fair coin

for letter, answer in zip("abcdefg", [answer_4_1a, answer_4_1b, answer_4_1c, answer_4_1d, answer_4_1e, answer_4_1f, answer_4_1g]):
    wb.check(f"4.1{letter}", answer)

**Ex 4.2 — Une pile : le verdict s'inverse**

In [ ]:
# Ex 4.2: the values you found on paper
answer_4_2a = ...  # a) P(tails | rigged), 2 decimals
answer_4_2b = ...  # b) the evidence P(tails), 3 decimals
answer_4_2c = ...  # c) P(fair | tails), 3 decimals
answer_4_2d = ...  # d) how much tails raises P(fair) above the prior 0.5, 3 decimals
answer_4_2e = ...  # e) P(fair | heads) P(heads) + P(fair | tails) P(tails), 3 decimals

for letter, answer in zip("abcde", [answer_4_2a, answer_4_2b, answer_4_2c, answer_4_2d, answer_4_2e]):
    wb.check(f"4.2{letter}", answer)

**Ex 4.4 — Vie extraterrestre : lire la sonde avec Bayes**

In [ ]:
# Ex 4.4: the values you found on paper
answer_4_4a = ...  # a) the sensitivity P(detected | inhabited), 2 decimals
answer_4_4b = ...  # b) the specificity P(nothing | sterile), 2 decimals
answer_4_4c = ...  # c) P(inhabited | nothing detected) in thousandths, 2 decimals
answer_4_4d = ...  # d) P(inhabited | detected), 3 decimals
answer_4_4e = ...  # e) the precision on the 2 000 test planets, 3 decimals
answer_4_4f = ...  # f) false alarms expected on 10 000 planets of the region

for letter, answer in zip("abcdef", [answer_4_4a, answer_4_4b, answer_4_4c, answer_4_4d, answer_4_4e, answer_4_4f]):
    wb.check(f"4.4{letter}", answer)

**Ex 4.5 — Deux faces : une mise à jour double ou deux simples ?**

In [ ]:
# Ex 4.5: the values you found on paper
answer_4_5a = ...  # a) P(fair | heads, heads) in two steps, 3 decimals
answer_4_5b = ...  # b) P(heads, heads | rigged), 4 decimals
answer_4_5c = ...  # c) the evidence P(heads, heads), 3 decimals
answer_4_5d = ...  # d) P(fair | heads, heads) at once, 3 decimals
answer_4_5e = ...  # e) P(fair | heads, heads, tails), 3 decimals
answer_4_5f = ...  # f) P(fair | tails, heads, heads), 3 decimals

for letter, answer in zip("abcdef", [answer_4_5a, answer_4_5b, answer_4_5c, answer_4_5d, answer_4_5e, answer_4_5f]):
    wb.check(f"4.5{letter}", answer)

**Ex 4.6 — Cinq hypothèses de biais après face, pile, face**

In [ ]:
# Ex 4.6: the values you found on paper
answer_4_6a = ...  # a) the posterior after heads, a list (2 decimals)
answer_4_6b = ...  # b) the posterior after heads, tails, a list (3 decimals)
answer_4_6c = ...  # c) the posterior after heads, tails, heads, a list (3 decimals)
answer_4_6d = ...  # d) the evidence P(heads, tails, heads), 4 decimals
answer_4_6e = ...  # e) the MAP hypothesis (a value of theta)

for letter, answer in zip("abcde", [answer_4_6a, answer_4_6b, answer_4_6c, answer_4_6d, answer_4_6e]):
    wb.check(f"4.6{letter}", answer)

**Ex 4.8 — Combien de sondes pour descendre sous un sur un million ?**

In [ ]:
# Ex 4.8: the values you found on paper
answer_4_8a = ...  # a) the prior odds P(inhabited) / P(sterile), 4 decimals
answer_4_8b = ...  # b) the factor that divides the odds after a negative probe, 1 decimal
answer_4_8c = ...  # c) P(inhabited | 2 negative probes) in millionths, 3 decimals
answer_4_8d = ...  # d) the minimal number of negative probes
answer_4_8e = ...  # e) P(inhabited | 2 positive probes), 3 decimals
answer_4_8f = ...  # f) P(inhabited | positive, then negative), 3 decimals

for letter, answer in zip("abcdef", [answer_4_8a, answer_4_8b, answer_4_8c, answer_4_8d, answer_4_8e, answer_4_8f]):
    wb.check(f"4.8{letter}", answer)

## Partie A · Des pièces, des fréquences et une première règle de Bayes

Fiche §4.2 à §4.4.2. Tu prévois combien de lancers il faut pour démasquer une pièce truquée, tu traces l'estimation fréquentiste du biais, puis tu écris les deux premières fonctions de `mylearn.bayes` et tu les appliques aux manchots. La cellule ci-dessous charge les données et les outils de tout le notebook : exécute-la d'abord.

In [ ]:
# Tools and data for the notebook exercises (parts A to D)
import inspect

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import special, stats

penguins = wb.datasets.load_penguins()           # the 344 penguins (4.15, 4.23)
SPECIES = ["Adelie", "Chinstrap", "Gentoo"]       # always in this order in the answers


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def fr(value, decimals=2):
    """A number written the French way, for the messages: fr(0.25) -> '0,25'."""
    return f"{value:.{decimals}f}".replace(".", ",")


def filled(*values):
    """True when none of the values is still `...` (or None): the answer has been written."""
    return all(value is not ... and value is not None for value in values)


def run_bayes_tests(keyword, impl="learner"):
    """Run the tests of mylearn.bayes selected by `keyword` (on YOUR code by default)."""
    command = [sys.executable, "-m", "pytest", "tests/test_ch04_bayes.py", "-k", keyword, "-q",
               "-p", "no:cacheprovider", "--color=no", "-rf", "--tb=no"]
    if impl != "learner":
        command.append(f"--impl={impl}")
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                            env={**os.environ, "COLUMNS": "1000"})   # long lines: the reason of each failure
    lines = result.stdout.strip().splitlines()
    failed = [line for line in lines if line.startswith("FAILED ")]
    for line in failed[:8]:                                  # the test, then the reason of its failure
        name, _, reason = line.removeprefix("FAILED tests/test_ch04_bayes.py::").partition(" - ")
        print(f"❌ {name}\n   {reason[:800]}")
    if len(failed) > 8:
        print(f"   ... and {len(failed) - 8} other failed test(s)")
    print("pytest:", lines[-1] if lines else result.stderr.strip()[-300:])


def error_name(func, *args, **kwargs):
    """Name of the exception raised by func(*args, **kwargs), or "no error"."""
    try:
        func(*args, **kwargs)
    except NotImplementedError:
        raise
    except Exception as error:  # noqa: BLE001 - we want the name of whatever is raised
        return type(error).__name__
    return "no error"


print(f"{len(penguins)} penguins")

### Ex 4.12 — Combien de lancers pour démasquer la pièce truquée ? 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir combien de lancers il faut pour distinguer deux pièces, selon l'écart entre leurs biais.  
**Prérequis :** Ex 4.1 · fiche §4.4.2 (🧮 cotes) · fais ta prédiction avant de lire le §4.6.2, du livre comme de la fiche · **Fil rouge :** synthétique · **Parcours :** C

Une pièce équilibrée et une pièce truquée, une chance sur deux chacune : tu prends l'une des deux et tu la lances, encore et encore. Après chaque lancer, tu mets à jour $P(\text{truquée})$ (la boucle que tu verras au §4.6 de la fiche) ; tu t'arrêtes dès que l'une des deux hypothèses dépasse 0,95, et tu annonces la plus probable. La fonction `flips_to_decide` ci-dessous fait exactement cela, avec la forme « cotes » de la fiche.

**Sans rien exécuter**, prévois :
a) `prediction_4_12a` : si la pièce lancée est la truquée et que son biais vaut 0,75, combien de lancers faut-il en général ? (la médiane sur 1 000 essais, un entier) ;
b) `prediction_4_12b` : même question si le biais de la pièce truquée ne vaut que 0,55 (un entier) ;
c) `prediction_4_12c` : avec la pièce de biais 0,75, sur 1 000 essais, combien de fois environ annonce-t-on la mauvaise pièce ? (un entier).

Écris ton hypothèse (cellule 📝), puis tes trois prédictions. Ensuite seulement, exécute l'**Expérience** : 1 000 essais pour chaque biais, avec la pièce truquée puis avec la pièce équilibrée. Rien n'est vérifié automatiquement ici : compare toi-même, puis note ce que tu retiens.

In [ ]:
def flips_to_decide(bias_rigged, true_bias, rng, threshold=0.95, max_flips=50_000):
    """Flip until P(rigged) or P(fair) exceeds `threshold`: (number of flips, coin announced)."""
    step_heads = np.log(bias_rigged / 0.5)           # what one heads adds to log(P(rigged) / P(fair))
    step_tails = np.log((1 - bias_rigged) / 0.5)     # (fiche, 🧮 cotes : les log-cotes s'additionnent)
    limit = np.log(threshold / (1 - threshold))      # the log-odds of the threshold
    log_odds, n = 0.0, 0                             # prior 0.5 each: odds 1, log-odds 0
    while abs(log_odds) < limit and n < max_flips:
        n += 1
        log_odds += step_heads if rng.random() < true_bias else step_tails
    return n, ("rigged" if log_odds > 0 else "fair")

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_4_12a = ...   # an integer
prediction_4_12b = ...   # an integer
prediction_4_12c = ...   # an integer

**Expérience** : exécute la cellule et compare avec tes prédictions (une dizaine de secondes).

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_4_12a, prediction_4_12b, prediction_4_12c]):
    print("⏳ Ex 4.12 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    rng_12 = np.random.default_rng(412)
    print(f"{'rigged coin':<13}{'coin flipped':<15}{'median flips':>13}{'wrong announcements':>21}")
    for bias in (0.75, 0.55):
        for truth in ("rigged", "fair"):
            runs = [flips_to_decide(bias, bias if truth == "rigged" else 0.5, rng_12) for _ in range(1000)]
            n_flips = np.array([n for n, _ in runs])
            wrong = sum(announced != truth for _, announced in runs)
            print(f"bias {bias:<8}{truth:<15}{np.median(n_flips):>13.0f}{wrong:>15} / 1000")
    print(f"your predictions: a) {prediction_4_12a}   b) {prediction_4_12b}   c) {prediction_4_12c}")

📝 **Ce que je retiens** (comparaison avec mes prédictions ; pourquoi le biais 0,55 demande-t-il autant de lancers ? d'où viennent les annonces fausses, et leur nombre était-il prévisible ?) : …

### Ex 4.13 — L'estimation fréquentiste : la moyenne courante des faces 🔬 ★ ⏱️ 15 min
**Objectif :** tracer l'estimation fréquentiste du biais après chaque lancer, et mesurer sa précision.  
**Prérequis :** ch. 2 (loi de Bernoulli, vitesse en $1/\sqrt{n}$) · fiche §4.3 · livre §4.2.1, §4.3 (figure 4.1) · **Fil rouge :** synthétique · **Parcours :** C

Le livre ouvre le chapitre avec trois pièces, de biais 0,2, 0,5 et 0,8 (figure 4.1) : après chaque lancer, il estime le biais par la proportion de faces obtenues jusque-là. C'est l'estimation **fréquentiste** du biais. Les 100 lancers de chaque pièce sont dans `flips_13` (1 = face, 0 = pile).

Écris `running_estimate(flips)`, qui renvoie le tableau des $n$ estimations : la $k$-ième vaut le nombre de faces parmi les $k$ premiers lancers, divisé par $k$ (pense à `np.cumsum` et `np.arange`). La vérification trace les trois courbes, comme la figure 4.1.

a) Vérifié avec ta fonction : les trois estimations après 100 lancers, dans l'ordre des biais 0,2, 0,5 et 0,8.
b) `max_error_13` : pour la pièce de biais 0,8, la plus grande erreur $|\text{estimation} - 0{,}8|$ entre le 50ᵉ et le 100ᵉ lancer compris (3 décimales).
c) `se_13` : l'erreur typique (l'écart-type) de l'estimation après $n = 100$ lancers, d'après le ch. 2 : $\sqrt{\theta(1 - \theta)/n}$ pour $\theta = 0{,}8$ (3 décimales).

Dans tes notes : l'erreur de b) est-elle grande ou petite par rapport à l'erreur typique de c) ? Combien de lancers faudrait-il pour diviser l'erreur typique par 10 ?

In [ ]:
flips_13 = {bias: wb.synth.coin_flips(100, bias, seed=int(100 * bias)) for bias in (0.2, 0.5, 0.8)}
print({bias: "".join(map(str, flips[:20])) + "..." for bias, flips in flips_13.items()})

In [ ]:
def running_estimate(flips):
    """Estimate of the bias after each flip: share of heads among the first k flips (k = 1, 2, ...)."""
    raise NotImplementedError("running_estimate() is not written yet")


max_error_13 = ...   # b)
se_13 = ...          # c)

In [ ]:
with wb.attempt("4.13"):
    curves_13 = {bias: np.asarray(running_estimate(flips), dtype=float) for bias, flips in flips_13.items()}
    right_13 = all(curve.shape == (100,) and np.allclose(curve, np.cumsum(flips_13[bias]) / np.arange(1, 101))
                   for bias, curve in curves_13.items())
    verdict("4.13", right_13, "running_estimate donne la proportion de faces après chaque lancer.",
            "running_estimate(flips)[k - 1] doit valoir le nombre de faces parmi les k premiers lancers, divisé par k "
            "(un tableau de la longueur de flips).")
    wb.check("4.13a", [curves_13[bias][-1] for bias in (0.2, 0.5, 0.8)], computed=True)
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
    for ax, (bias, curve) in zip(axes, curves_13.items()):
        ax.plot(np.arange(1, 101), curve, color="C0")
        ax.axhline(bias, color="C1", ls="--", lw=1)
        ax.set_title(f"coin of bias {bias}")
        ax.set_xlabel("number of flips")
    axes[0].set_ylabel("running estimate of the bias")
    plt.show()
wb.check("4.13b", max_error_13)
wb.check("4.13c", se_13)

### Ex 4.14 — evidence et bayes_posterior 🔨 ★★ ⏱️ 20 min
**Objectif :** écrire l'évidence et la règle de Bayes pour un nombre fini d'hypothèses, avec leurs contrôles.  
**Prérequis :** Ex 4.1 · fiche §4.4.1, §4.4.2 · **Fil rouge :** synthétique · **mylearn :** `bayes.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/bayes.py` (créé par `python tools/start_chapter.py 4`), lis la docstring de chaque fonction, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy est permis (`np.asarray`, `np.dot`, `np.sum`, `np.log`, `np.exp`, `np.cumsum`…), SciPy non : `scipy.special` et `scipy.stats` sont les **oracles** des tests. Écris une petite fonction d'aide, par exemple `_check_distribution(p, name)`, qui convertit en tableau de flottants (`np.asarray(p, dtype=float)`), lève une `ValueError` s'il n'est pas à une dimension, s'il est vide, s'il contient une valeur négative ou un NaN, ou si sa somme s'écarte de 1 de plus de `1e-8`, et le renvoie : les fonctions du module l'appelleront. Teste les NaN à part, avec `np.isnan` : `nan < 0` vaut `False`, et une somme qui contient un NaN échappe aussi à la comparaison avec 1. La cellule de vérification recharge ta librairie, vérifie quelques valeurs, puis lance les tests de tes fonctions ; `python -m pytest tests/test_ch04_bayes.py -q` les lance tous dans un terminal.

Écris `evidence(prior, likelihood)` et `bayes_posterior(prior, likelihood)` (lis leurs docstrings) :
- valide les deux entrées : le prior est une distribution (valeurs positives ou nulles, de somme 1 à `1e-8` près) ; les vraisemblances sont des probabilités (entre 0 et 1), sans obligation de sommer à 1 ; les deux ont la même forme (attention : le broadcasting de NumPy étirerait sans rien dire un tableau de longueur 1) ;
- `evidence` renvoie un `float` Python, la somme des produits vraisemblance × prior ;
- `bayes_posterior` réutilise `evidence`, lève une `ValueError` si elle vaut 0 (l'observation est impossible sous toutes les hypothèses), puis renvoie le tableau des produits divisés par l'évidence, sans modifier les tableaux reçus.

Vérifications, sur un sac de trois pièces : une pièce équilibrée (prior 0,5), une de biais 0,3 (prior 0,25) et une de biais 0,9 (prior 0,25). On tire une pièce et on la lance : **pile**. La cellule de vérification appelle tes fonctions :
a) l'évidence $P(\text{pile})$ ;
b) le posterior des trois pièces ;
puis deux contrôles d'erreurs et les tests des deux fonctions.

In [ ]:
prior_14 = np.array([0.5, 0.25, 0.25])            # fair coin, coin of bias 0.3, coin of bias 0.9
lik_tails_14 = 1 - np.array([0.5, 0.3, 0.9])      # P(tails | each coin)

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="4")   # reload your saved file
with wb.attempt("4.14"):
    wb.check("4.14a", mylearn.bayes.evidence(prior_14, lik_tails_14), computed=True)
    wb.check("4.14b", mylearn.bayes.bayes_posterior(prior_14, lik_tails_14), computed=True)
    verdict("4.14", error_name(mylearn.bayes.bayes_posterior, [0.5, 0.5], [0.0, 0.0]) == "ValueError",
            "une observation impossible (évidence nulle) lève bien une ValueError.",
            "bayes_posterior([0.5, 0.5], [0.0, 0.0]) doit lever une ValueError : l'évidence est nulle.")
    verdict("4.14", error_name(mylearn.bayes.evidence, [0.5, 0.6], [0.5, 0.5]) == "ValueError",
            "un prior qui ne somme pas à 1 est refusé.",
            "evidence([0.5, 0.6], [0.5, 0.5]) doit lever une ValueError : ce prior somme à 1,1.")
    run_bayes_tests("test_evidence_ or test_bayes_posterior_")

### Ex 4.15 — Bayes chez les manchots : l'espèce sachant l'île 📦 ★★ ⏱️ 20 min
**Objectif :** retrouver une probabilité conditionnelle d'une table de contingence avec la règle de Bayes, puis la recalculer pour une autre population.  
**Prérequis :** Ex 4.14 · ch. 3 (Ex 3.14, `pd.crosstab`) · fiche §4.4.1, §4.5 · **Fil rouge :** Penguins · **Parcours :** R, C

Au ch. 3 (3.14), tu as lu $P(\text{espèce} \mid \text{île})$ directement dans la table espèce × île. Retrouve-la avec la règle de Bayes, en séparant ses deux ingrédients : la composition de la population (le **prior**) et l'endroit où vit chaque espèce (la **vraisemblance**).

Les hypothèses sont les trois espèces, toujours dans l'ordre `SPECIES` (Adelie, Chinstrap, Gentoo) ; l'observation est « le manchot a été vu sur l'île Dream ». Pour les 344 manchots de `penguins` :
a) `prior_15` : $P(\text{espèce})$, un tableau de trois probabilités (vérifié à 3 décimales, mais **n'arrondis pas** : c) à e) le réutilisent) ;
b) `likelihood_15` : $P(\text{Dream} \mid \text{espèce})$, un tableau de trois probabilités (vérifié à 3 décimales, sans arrondir non plus ; `pd.crosstab` et son argument `normalize` t'y aident) ;
c) vérifié avec ta fonction `bayes_posterior` : $P(\text{espèce} \mid \text{Dream})$ ; la vérification la compare à la colonne Dream de `pd.crosstab(..., normalize="columns")` ;
d) vérifié avec ta fonction `evidence` : $P(\text{Dream})$ ;
e) une autre équipe estime que, dans tout l'archipel, 60 % des manchots sont des Adélie, 15 % des Chinstrap et 25 % des Gentoo. Écris ce prior dans `other_prior_15` : la vérification calcule avec ta fonction $P(\text{espèce} \mid \text{Dream})$ pour cette population, avec les mêmes vraisemblances.

Dans tes notes : pourquoi peut-on garder les vraisemblances $P(\text{île} \mid \text{espèce})$ d'une population à l'autre, mais pas le prior ? Quand cette hypothèse serait-elle fausse ?

In [ ]:
prior_15 = ...         # a) P(species), order Adelie, Chinstrap, Gentoo
likelihood_15 = ...    # b) P(Dream | species), same order
other_prior_15 = ...   # e) the other team's prior

In [ ]:
wb.check("4.15a", prior_15)
wb.check("4.15b", likelihood_15)
if not filled(prior_15, likelihood_15):
    print("⏳ Ex 4.15 c-d : remplis d'abord a) et b).")
else:
    with wb.attempt("4.15"):
        posterior_15 = mylearn.bayes.bayes_posterior(prior_15, likelihood_15)
        wb.check("4.15c", posterior_15, computed=True)
        wb.check("4.15d", mylearn.bayes.evidence(prior_15, likelihood_15), computed=True)
        dream_15 = pd.crosstab(penguins["species"], penguins["island"], normalize="columns")["Dream"].reindex(SPECIES)
        verdict("4.15", np.allclose(posterior_15, dream_15.to_numpy()),
                "Bayes redonne exactement la colonne Dream de pd.crosstab(..., normalize=\"columns\").",
                "ton posterior diffère de la colonne Dream de pd.crosstab(..., normalize=\"columns\") : vérifie a) et b), "
                "et ne les arrondis pas.")
if not filled(other_prior_15, likelihood_15):
    print("⏳ Ex 4.15 e : remplis d'abord b) et other_prior_15.")
else:
    with wb.attempt("4.15"):
        wb.check("4.15e", mylearn.bayes.bayes_posterior(other_prior_15, likelihood_15), computed=True)

## Partie B · La boucle posterior → prior

Fiche §4.5 à §4.6.2 et « au-delà du livre (1) ». Tu écris la mise à jour séquentielle (`update_discrete`), tu refais les barres empilées du livre, tu observes un underflow et tu le corriges, puis tu balaies une grille de pièces et tu envoies des sondes jusqu'à la décision.

### Ex 4.16 — La boucle posterior-prior : update_discrete 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire la mise à jour séquentielle d'une distribution sur des hypothèses, observation après observation.  
**Prérequis :** Ex 4.14 · fiche §4.6, §4.6.1 · **Fil rouge :** synthétique · **mylearn :** `bayes.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/bayes.py`, mêmes règles qu'en 4.14 (NumPy permis, SciPy non). Enregistre, puis relance la cellule de vérification.

Écris `update_discrete(prior, likelihoods, observations, return_history=False)` (lis sa docstring) :
- `likelihoods` est un **tableau** : une ligne par hypothèse, une colonne par issue possible, `likelihoods[i, o]` $= P(\text{issue } o \mid H_i)$. Pour une pièce, deux colonnes (0 = pile, 1 = face) ;
- `observations` contient des **indices d'issues** ; pour chacune, la colonne correspondante donne les vraisemblances, la règle de Bayes (ta fonction `bayes_posterior`) donne le posterior, et ce posterior devient le prior de l'observation suivante ;
- valide tout : un tableau à deux dimensions avec une ligne par hypothèse, des indices entiers entre 0 et le nombre d'issues moins 1 (attention : NumPy accepterait l'indice −1 sans rien dire, il lit alors la dernière colonne) ;
- sans aucune observation (`[]`), elle renvoie le prior ; avec `return_history=True`, elle renvoie un tableau dont la ligne 0 est le prior et la ligne $k$ le posterior après les $k$ premières observations.

Vérifications, sur trois dés possibles (`DICE_16`) : un dé équilibré, un dé pipé sur le 6 et un dé pipé sur le 1, avec le prior `prior_16`. On lance le dé cinq fois : les faces sont dans `rolls_16` (la face $k$ est l'issue d'indice $k - 1$). La cellule de vérification appelle ta fonction :
a) le posterior final des trois dés ;
b) le posterior après les deux premiers lancers (la ligne 2 de l'historique) ;
c) `most_likely_16` : après les cinq lancers, quel dé est le plus probable ? Réponds par son numéro de ligne (0, 1 ou 2), en lisant le résultat que la vérification affiche.

Puis un contrôle (l'ordre des lancers ne change rien), un contrôle d'erreur et les tests de `update_discrete`.

In [ ]:
DICE_16 = np.array([[1 / 6] * 6,                       # 0: a fair die
                    [0.1, 0.1, 0.1, 0.1, 0.1, 0.5],    # 1: a die loaded on the 6
                    [0.5, 0.1, 0.1, 0.1, 0.1, 0.1]])   # 2: a die loaded on the 1
prior_16 = np.array([0.8, 0.1, 0.1])
rolls_16 = np.array([6, 6, 2, 6, 5])                   # the faces rolled

In [ ]:
most_likely_16 = ...   # c) 0, 1 or 2

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="4")   # reload your saved file
with wb.attempt("4.16"):
    final_16 = mylearn.bayes.update_discrete(prior_16, DICE_16, rolls_16 - 1)
    history_16 = np.asarray(mylearn.bayes.update_discrete(prior_16, DICE_16, rolls_16 - 1, return_history=True))
    print("final posterior:", np.round(final_16, 4))
    wb.check("4.16a", final_16, computed=True)
    wb.check("4.16b", history_16[2] if history_16.ndim == 2 and len(history_16) > 2 else history_16, computed=True)
    reversed_16 = mylearn.bayes.update_discrete(prior_16, DICE_16, (rolls_16 - 1)[::-1])
    verdict("4.16", np.allclose(reversed_16, final_16), "les mêmes lancers dans l'ordre inverse donnent le même posterior.",
            "dans l'ordre inverse, les mêmes lancers doivent donner le même posterior final.")
    verdict("4.16", error_name(mylearn.bayes.update_discrete, prior_16, DICE_16, [6]) == "ValueError",
            "l'issue 6 (une septième face) lève bien une ValueError.",
            "update_discrete(prior_16, DICE_16, [6]) doit lever une ValueError : les issues vont de 0 à 5.")
    run_bayes_tests("test_update_discrete_")
wb.check("4.16c", most_likely_16)

### Ex 4.17 — Reproduire les trente lancers de la figure 4.24 🎨 ★★ ⏱️ 25 min
**Objectif :** tracer, lancer après lancer, la probabilité de chacune de deux pièces en barres empilées.  
**Prérequis :** Ex 4.16 · fiche §4.6.2 · livre §4.6.2 (figures 4.24 à 4.26) · **Fil rouge :** synthétique · **Parcours :** C

Le livre suit, lancer après lancer, la probabilité d'avoir la pièce équilibrée plutôt qu'une pièce de biais 0,2 (prior 0,5 chacune), en barres empilées (figures 4.24 à 4.26). Refais ses trois figures avec les trois suites de 30 lancers de `SEQUENCES_17` (des chaînes de caractères : `"F"` pour face, `"P"` pour pile), qui contiennent 6, 3 et 24 faces.

1. Écris `stacked_history_17(sequence)` : elle convertit la chaîne en tableau d'issues (1 pour `"F"`, 0 pour `"P"`), puis renvoie l'historique de `mylearn.bayes.update_discrete`, un tableau de forme `(31, 2)` dont la colonne 0 est $P(\text{équilibrée})$ et la colonne 1 $P(\text{truquée})$.
2. Écris `draw_stacked_17(ax, sequence)` : sur l'axe `ax`, des barres de hauteur $P(\text{équilibrée})$, puis, empilées au-dessus (`bottom=`), des barres de hauteur $P(\text{truquée})$ ; en abscisse, « avant » puis la lettre de chaque lancer.

La vérification contrôle ton historique, puis trace les trois panneaux avec ta fonction. Compare avec le livre, puis note dans tes notes après combien de lancers la pièce truquée dépasse 0,9 dans la suite à 3 faces.

In [ ]:
SEQUENCES_17 = {"6 heads": "PPPPPFFFPPPPPPPPPPFPPPFFPPPPPP",     # F = face (heads), P = pile (tails)
                "3 heads": "PPPPFPPPPPPPPPPFPPPPPPPPPFPPPP",
                "24 heads": "PFFPFFFFFPFFFFFFFFFFPFFFPFFFPF"}
COINS_17 = np.array([[0.5, 0.5],      # fair coin: P(tails), P(heads)
                     [0.8, 0.2]])     # rigged coin, bias 0.2

In [ ]:
def stacked_history_17(sequence):
    """(31, 2) array: P(fair) and P(rigged) before the flips, then after each flip of `sequence`."""
    raise NotImplementedError("stacked_history_17() is not written yet")


def draw_stacked_17(ax, sequence):
    """Stacked bars of P(fair) (bottom) and P(rigged) (top), one bar before the flips and one per flip."""
    raise NotImplementedError("draw_stacked_17() is not written yet")

In [ ]:
with wb.attempt("4.17"):
    ok_17 = True
    for sequence in SEQUENCES_17.values():
        flips = np.array([1 if c == "F" else 0 for c in sequence])
        heads, tails = np.r_[0, np.cumsum(flips)], np.r_[0, np.cumsum(1 - flips)]
        odds = (0.2 / 0.5) ** heads * (0.8 / 0.5) ** tails          # P(rigged) / P(fair) after each flip
        expected = np.column_stack([1 / (1 + odds), odds / (1 + odds)])
        got = np.asarray(stacked_history_17(sequence), dtype=float)
        ok_17 = ok_17 and got.shape == (31, 2) and np.allclose(got, expected)
    verdict("4.17", ok_17, "tes trois historiques sont justes : à toi de juger la figure.",
            "stacked_history_17 doit renvoyer un tableau (31, 2) : colonne 0 = P(équilibrée), colonne 1 = P(truquée), "
            "ligne 0 = le prior [0,5 ; 0,5].")
    fig, axes = plt.subplots(3, 1, figsize=(12, 7.5), sharex=True)
    for ax, (label, sequence) in zip(axes, SEQUENCES_17.items()):
        draw_stacked_17(ax, sequence)
        ax.set_title(f"30 flips, {label} (rigged coin: bias 0.2)", fontsize=10)
    plt.tight_layout()
    plt.show()

### Ex 4.18 — Le posterior qui s'évanouit : underflow 🐛 ★★ ⏱️ 20 min
**Objectif :** diagnostiquer un posterior qui devient `nan`, et le corriger en passant aux log-probabilités.  
**Prérequis :** Ex 4.16 · fiche, au-delà du livre (1) (🧮 underflow, log-sum-exp) · 0B (logarithmes) · **Fil rouge :** synthétique · **Parcours :** R, M, C

Un collègue estime le biais d'une pièce parmi trois candidats (0,58 ; 0,60 ; 0,62), avec la fonction `posterior_after_flips` ci-dessous : elle multiplie toutes les vraisemblances, puis normalise une seule fois à la fin. Sur 200 lancers, tout va bien ; sur 3 000, elle renvoie `[nan nan nan]`.

a) `n_zero_18` : le plus petit entier $n$ pour lequel `0.5 ** n == 0.0` en Python (cherche-le avec une boucle) ;
b) `first_nan_18` : le plus petit nombre de lancers $k$ pour lequel `posterior_after_flips(prior_18, biases_18, flips_18[:k])` contient des `nan`. Inutile de l'appeler 3 000 fois : calcule d'un coup, avec `np.cumprod`, le tableau `joint` de la fonction (le prior multiplié par le produit des vraisemblances) après chaque lancer, et cherche quand il devient exactement 0 pour **chacune** des trois hypothèses ;
c) corrige : écris `posterior_fixed_18(prior, biases, flips)`, avec le même contrat mais en log-probabilités. Additionne $\log P(\theta)$, $h \log\theta$ et $t \log(1 - \theta)$, retranche le maximum, puis prends l'exponentielle et normalise (fiche, 🧮 log-sum-exp). La vérification compare ta version à celle du collègue sur 200 lancers, puis vérifie ton posterior après les 3 000 lancers ;
d) `lse_18` : sans SciPy, avec l'astuce du maximum, $\log(e^{-1000} + e^{-1001})$ (3 décimales).

Dans tes notes : pourquoi le calcul direct de d) échoue-t-il ? Pourquoi aucune erreur n'apparaît-elle au moment où le produit devient 0 ?

In [ ]:
def posterior_after_flips(prior, biases, flips):
    """Posterior over the candidate biases after the flips (the colleague's version)."""
    prior, biases = np.asarray(prior, dtype=float), np.asarray(biases, dtype=float)
    likelihood = np.ones(len(biases))                  # P(all the flips so far | each bias)
    for flip in flips:
        likelihood = likelihood * (biases if flip == 1 else 1 - biases)
    joint = prior * likelihood
    return joint / joint.sum()


biases_18 = np.array([0.58, 0.60, 0.62])
prior_18 = np.full(3, 1 / 3)
flips_18 = wb.synth.coin_flips(3000, 0.6, seed=25)
with np.errstate(invalid="ignore"):                    # silence the 0 / 0 warning
    print("after 200 flips: ", posterior_after_flips(prior_18, biases_18, flips_18[:200]).round(4))
    print("after 3000 flips:", posterior_after_flips(prior_18, biases_18, flips_18))

In [ ]:
n_zero_18 = ...       # a)
first_nan_18 = ...    # b)


def posterior_fixed_18(prior, biases, flips):
    """Same contract as posterior_after_flips, computed with log-probabilities (no underflow)."""
    raise NotImplementedError("posterior_fixed_18() is not written yet")


lse_18 = ...          # d)

In [ ]:
wb.check("4.18a", n_zero_18)
wb.check("4.18b", first_nan_18)
with wb.attempt("4.18"):
    short_18 = np.asarray(posterior_fixed_18(prior_18, biases_18, flips_18[:200]), dtype=float)
    verdict("4.18", short_18.shape == (3,) and np.allclose(short_18, posterior_after_flips(prior_18, biases_18, flips_18[:200])),
            "sur 200 lancers, ta version et celle du collègue sont d'accord.",
            "sur 200 lancers (pas d'underflow), ta version doit redonner celle du collègue.")
    wb.check("4.18c", posterior_fixed_18(prior_18, biases_18, flips_18), computed=True)
wb.check("4.18d", lse_18)

### Ex 4.19 — La grille biais × proportion de faces 🔬 ★★ ⏱️ 30 min
**Objectif :** mesurer, sur une grille de pièces et de séries de lancers, quand Bayes sait trancher et quand il hésite.  
**Prérequis :** Ex 4.16 · fiche §4.6.2 · livre §4.6.2 (figures 4.27 et 4.28) · **Fil rouge :** synthétique · **Parcours :** C

Le livre balaie une grille (figures 4.27 et 4.28) : en abscisse le biais de la pièce truquée, en ordonnée la proportion de faces d'une série de lancers inventée ; dans chaque case, la probabilité finale que la pièce soit l'équilibrée (prior 0,5 chacune). Refais-la avec les dix proportions `PROPORTIONS_19` et les dix biais `BIASES_19` (0,05 ; 0,15 ; … ; 0,95).

Écris `grid_19(n)`, qui renvoie un tableau $10 \times 10$ : la case `[i, j]` vaut $P(\text{équilibrée} \mid \text{lancers})$ pour une série de $n$ lancers contenant `round(PROPORTIONS_19[i] * n)` faces, quand la pièce truquée a le biais `BIASES_19[j]`. Le posterior ne dépend que du nombre de faces (4.16) : prends la série `[1] * h + [0] * (n - h)` et ta fonction `update_discrete`.

a) vérifié avec ta fonction : `grid_19(40)` ;
b) `decided_19` : une case est « tranchée » si $P(\text{équilibrée})$ y est inférieure à 0,05 ou supérieure à 0,95. Combien de cases sont tranchées pour $n = 40$, puis pour $n = 1\,000$ ? (une liste de deux entiers).

La vérification trace les deux grilles (ligne 0 en bas). Dans tes notes : où se trouvent les cases indécises ? Que se passe-t-il dans le coin où l'on observe 95 % de faces avec une pièce truquée de biais 0,05 ? Est-ce rassurant ?

In [ ]:
PROPORTIONS_19 = (2 * np.arange(10) + 1) / 20    # 0.05, 0.15, ..., 0.95: share of heads of the flips (rows)
BIASES_19 = PROPORTIONS_19.copy()                 # bias of the rigged coin (columns)

In [ ]:
def grid_19(n):
    """10 x 10 array: P(fair | n flips with round(p * n) heads); row = proportion p, column = rigged bias."""
    raise NotImplementedError("grid_19() is not written yet")


decided_19 = ...   # b) [number of decided cells for n = 40, for n = 1000]

In [ ]:
with wb.attempt("4.19"):
    grids_19 = {n: np.asarray(grid_19(n), dtype=float) for n in (40, 1000)}
    wb.check("4.19a", grids_19[40], computed=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    for ax, (n, grid) in zip(axes, grids_19.items()):
        image = ax.imshow(grid, origin="lower", cmap=wb.plot.diverging_cmap(), vmin=0, vmax=1)
        ax.set_xticks(range(10), [f"{b:.2f}" for b in BIASES_19], rotation=90)
        ax.set_yticks(range(10), [f"{p:.2f}" for p in PROPORTIONS_19])
        ax.set_xlabel("bias of the rigged coin")
        ax.set_ylabel("share of heads in the flips")
        ax.set_title(f"P(fair | {n} flips)")
    fig.colorbar(image, ax=axes, shrink=0.8)
    plt.show()
wb.check("4.19b", decided_19)

### Ex 4.20 — Envoyer des sondes jusqu'à la décision 🔬 ★★ ⏱️ 25 min
**Objectif :** simuler une décision séquentielle : envoyer des sondes une par une jusqu'à être assez sûr, et compter les erreurs.  
**Prérequis :** Ex 4.4, Ex 4.16 · fiche §4.5, §4.6.1 · **Fil rouge :** synthétique · **Parcours :** C

On reprend la sonde Argos-2 de l'exercice 4.4 (sa sensibilité `SENS_20` et sa spécificité `SPEC_20` viennent de son tableau de test), dans la région où 5 % des planètes sont habitées. Pour chaque planète, le capitaine envoie des sondes **une par une**. Après chaque sonde, il met à jour $P(\text{habitée})$ ; il décide d'exploiter la planète dès que $P(\text{habitée}) < 10^{-6}$, et de la protéger dès que $P(\text{habitée}) > 0{,}99$. Les sondes sont supposées indépendantes sachant l'état de la planète.

Écris `explore_20(inhabited, rng)`, qui renvoie `(decision, probes)` : `"mine"` ou `"protect"`, et le nombre de sondes envoyées (au plus 20 ; au-delà, `"undecided"`). Pour que tout le monde obtienne les mêmes nombres, **tire exactement ainsi** : un seul `rng.random()` par sonde, au moment où tu l'envoies (ne tire pas les vingt d'avance) ; `detected = rng.random() < SENS_20` si la planète est habitée, `rng.random() < 1 - SPEC_20` sinon. Mets à jour avec ta fonction `update_discrete` et le tableau `TABLE_20` (une ligne par hypothèse : habitée, stérile ; une colonne par issue : 0 = rien, 1 = détecté, et `int(detected)` donne l'issue), en partant du prior `[PRIOR_20, 1 - PRIOR_20]`. La première sonde envoyée compte pour 1.

La vérification simule 10 000 planètes avec `simulate_20(explore_20)`, qui tire d'abord l'état de chaque planète, puis appelle ta fonction avec le même générateur. Vérifié avec ta fonction :
a) le nombre moyen de sondes par planète ;
b) le nombre de planètes protégées ;
c) le nombre de planètes stériles protégées par erreur.

Dans tes notes : combien de planètes habitées ont été exploitées ? Pourquoi ce nombre était-il prévisible ? Que deviendrait-il si deux sondes avaient tendance à se tromper ensemble (4.8 g) ?

In [ ]:
SENS_20, SPEC_20 = 240 / 250, 1645 / 1750         # the probe Argos-2 of 4.4 (from its test table)
PRIOR_20 = 0.05                                   # the same region
TABLE_20 = np.array([[1 - SENS_20, SENS_20],      # inhabited planet: P(nothing), P(detected)
                     [SPEC_20, 1 - SPEC_20]])     # sterile planet:   P(nothing), P(detected)


def simulate_20(explore, n_planets=10_000, seed=20):
    """Explore n_planets planets with `explore`; one generator for everything, in a fixed order."""
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(n_planets):
        inhabited = bool(rng.random() < PRIOR_20)        # one draw for the planet...
        decision, probes = explore(inhabited, rng)       # ...then one draw per probe, inside explore
        rows.append((inhabited, decision, probes))
    return pd.DataFrame(rows, columns=["inhabited", "decision", "probes"])

In [ ]:
def explore_20(inhabited, rng, low=1e-6, high=0.99, max_probes=20):
    """Send probes one by one until a decision: ("mine", "protect" or "undecided", number of probes sent)."""
    raise NotImplementedError("explore_20() is not written yet")

In [ ]:
with wb.attempt("4.20"):
    planets_20 = simulate_20(explore_20)
    wb.check("4.20a", planets_20["probes"].mean(), computed=True)
    wb.check("4.20b", int((planets_20["decision"] == "protect").sum()), computed=True)
    wb.check("4.20c", int(((planets_20["decision"] == "protect") & ~planets_20["inhabited"]).sum()), computed=True)
    print(f"probes per planet: mean {planets_20['probes'].mean():.3f}, at most {planets_20['probes'].max()}")
    print(pd.crosstab(planets_20["inhabited"], planets_20["decision"], margins=True))

## Partie C · Plusieurs hypothèses, un prior trompeur, la loi Beta

Fiche §4.7 et « au-delà du livre (2) ». Tu suis 501 hypothèses sur le biais d'une pièce avec un prior trompeur, tu retrouves la loi Beta avec `scipy.stats`, puis tu transformes un code copié-collé en une fonction documentée et testée.

### Ex 4.21 — Un prior trompeur centré sur 0,8 🔮 ★★ ⏱️ 20 min
**Objectif :** prévoir à quelle vitesse les données corrigent un prior trompeur, puis le mesurer.  
**Prérequis :** Ex 4.16 · fiche §4.7 · livre §4.7 (figures 4.33 à 4.35) · **Fil rouge :** synthétique · **Parcours :** M, C

On suit 501 hypothèses sur le biais d'une pièce, $\theta = 0 ;\ 0{,}002 ;\ \ldots ;\ 1$ (`GRID_21`). Le prior `PRIOR_21` est une bosse **trompeuse** : centrée sur 0,8, d'écart-type 0,1, elle ne laisse qu'une probabilité minuscule (mais jamais nulle) aux biais proches de 0,3. Or la pièce lancée a un biais de 0,3 ; ses 3 000 lancers sont dans `FLIPS_21`. **Sans rien exécuter**, prévois :
a) `prediction_4_21a` : après 100 lancers, le MAP (l'hypothèse de plus grand posterior) sera-t-il plus près de 0,3 que de 0,8 ? (`True` ou `False`) ;
b) `prediction_4_21b` : à partir de combien de lancers environ le MAP restera-t-il à moins de 0,015 de 0,3 jusqu'au 3 000ᵉ lancer ? Choisis l'ordre de grandeur : `10`, `100`, `1000` ou `10000` ;
c) `prediction_4_21c` : après 3 000 lancers, le posterior sera-t-il **exactement** celui qu'aurait donné un prior uniforme ? (`True` ou `False`).

Écris ton hypothèse (cellule 📝), puis tes trois prédictions ; la vérification ne regarde que tes prédictions. Ensuite seulement, exécute l'**Expérience** : ta fonction `update_discrete` sur les 3 000 lancers.

In [ ]:
GRID_21 = np.linspace(0, 1, 501)                                   # 501 hypotheses on the bias
PRIOR_21 = np.exp(-((GRID_21 - 0.8) ** 2) / (2 * 0.1 ** 2))          # a bump centred on 0.8...
PRIOR_21 = PRIOR_21 / PRIOR_21.sum()                                # ...normalized
TABLE_21 = np.column_stack([1 - GRID_21, GRID_21])                  # columns: P(tails | θ), P(heads | θ)
FLIPS_21 = wb.synth.coin_flips(3000, 0.3, seed=21)                  # the coin really has a bias of 0.3

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_4_21a = ...   # True or False
prediction_4_21b = ...   # 10, 100, 1000 or 10000
prediction_4_21c = ...   # True or False

In [ ]:
wb.check("4.21a", prediction_4_21a)
wb.check("4.21b", prediction_4_21b)
wb.check("4.21c", prediction_4_21c)

**Expérience** : exécute la cellule (quelques secondes) et compare avec tes prédictions.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_4_21a, prediction_4_21b, prediction_4_21c]):
    print("⏳ Ex 4.21 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    with wb.attempt("4.21"):
        history_21 = np.asarray(mylearn.bayes.update_discrete(PRIOR_21, TABLE_21, FLIPS_21, return_history=True))
        maps_21 = GRID_21[history_21.argmax(axis=1)]           # the MAP after 0, 1, ..., 3000 flips
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        for n in (0, 10, 50, 100, 300, 1000, 3000):
            axes[0].plot(GRID_21, history_21[n], label=f"{n} flips")
        axes[0].set_xlabel("bias θ (501 hypotheses)")
        axes[0].set_ylabel("posterior probability")
        axes[0].legend(fontsize=8)
        axes[1].plot(np.arange(len(maps_21)), maps_21, color="C0", label="MAP")
        axes[1].axhline(0.3, color="C1", ls="--", lw=1, label="true bias 0.3")
        axes[1].set_xscale("symlog", linthresh=10)
        axes[1].set_xlim(0, len(maps_21) - 1)
        axes[1].set_xlabel("number of flips (log scale after 10)")
        axes[1].set_ylabel("MAP")
        axes[1].legend(fontsize=8)
        plt.show()

Avec `history_21` (une ligne par nombre de lancers, de 0 à 3 000) et `maps_21` (le MAP après chaque nombre de lancers), calculés par l'expérience :
d) `map_100_21` : le MAP après 100 lancers (3 décimales) ;
e) `stays_21` : le plus petit nombre de lancers $n$ tel que, pour tous les nombres de lancers de $n$ à 3 000, le MAP est à moins de 0,015 de 0,3 (`np.abs(maps_21 - 0.3) < 0.015`).

In [ ]:
map_100_21 = ...   # d)
stays_21 = ...     # e)

In [ ]:
wb.check("4.21d", map_100_21)
wb.check("4.21e", stays_21)

### Ex 4.22 — Le posterior continu : vérifier avec scipy.stats.beta 📦 ★★ ⏱️ 20 min
**Objectif :** reconnaître la loi Beta dans le posterior d'une pièce, et la résumer avec `scipy.stats`.  
**Prérequis :** Ex 4.21 · fiche, au-delà du livre (2) (🧮 loi Beta) · ch. 2 (densité) · **Fil rouge :** synthétique · **Parcours :** M, C

Une pièce ancienne, lancée 20 fois, a donné 13 faces. Avec un prior **uniforme** sur $[0, 1]$, son posterior est une loi Beta (fiche, au-delà du livre (2)). Avec `scipy.stats.beta` :
a) `ab_22` : les deux paramètres $[a, b]$ de ce posterior (une liste de deux entiers) ;
b) `mean_22` : la moyenne du posterior (3 décimales) ;
c) `p_above_half_22` : $P(\theta > 0{,}5 \mid \text{données})$ (3 décimales) ;
d) `interval_22` : l'intervalle de crédibilité à 95 % à queues égales, `[basse, haute]` (3 décimales ; une méthode de la loi le donne directement).

La vérification contrôle ensuite, avec ta fonction `update_discrete` et un prior uniforme sur 1 001 points (`np.linspace(0, 1, 1001)`), que le posterior sur la grille est exactement la densité Beta normalisée sur la grille. Dans tes notes : pourquoi la moyenne n'est-elle pas égale à $\frac{13}{20}$ ?

In [ ]:
HEADS_22, TAILS_22 = 13, 7
GRID_22 = np.linspace(0, 1, 1001)

In [ ]:
ab_22 = ...             # a) [a, b]
mean_22 = ...           # b)
p_above_half_22 = ...   # c)
interval_22 = ...       # d) [low, high]

In [ ]:
for letter, answer in zip("abcd", [ab_22, mean_22, p_above_half_22, interval_22]):
    wb.check(f"4.22{letter}", answer)
with wb.attempt("4.22"):
    flips_22 = np.array([1] * HEADS_22 + [0] * TAILS_22)
    grid_post_22 = np.asarray(mylearn.bayes.update_discrete(np.full(1001, 1 / 1001), np.column_stack([1 - GRID_22, GRID_22]), flips_22))
    density_22 = stats.beta(HEADS_22 + 1, TAILS_22 + 1).pdf(GRID_22)
    gap_22 = float(np.max(np.abs(grid_post_22 - density_22 / density_22.sum())))
    verdict("4.22", gap_22 < 1e-10, f"le posterior sur la grille est la loi Beta normalisée (écart maximal {gap_22:.1e}).",
            f"le posterior sur la grille s'écarte de la densité Beta normalisée (écart maximal {gap_22:.1e}).")

### Ex 4.23 — Refactoriser : du copier-coller à une fonction testée 🛠️ ★★ ⏱️ 20 min
**Objectif :** remplacer un code copié-collé par une fonction documentée, et la protéger par des tests qui attrapent les erreurs classiques.  
**Prérequis :** Ex 4.15 · 0A.61, 0A.62 (docstring, pytest) · ch. 2 (2.31) · **Fil rouge :** Penguins · **Parcours :** R, C

Un collègue calcule $P(\text{espèce} \mid \text{île})$ pour chaque île en recopiant trois fois le même bloc (cellule ci-dessous). C'est ce qu'on appelle du **copier-coller** : chaque copie peut diverger, et c'est arrivé.

a) `buggy_island_23` : lis les trois blocs : lequel est faux ? (le nom de l'île, une chaîne). Dans tes notes : quelle erreur, et quel contrôle simple, fait sur les résultats affichés, l'aurait révélée sans relire le code ?

**Refactoriser**, c'est réécrire du code sans changer ce qu'il fait (sauf le bug !), pour qu'il soit plus simple et plus sûr. Écris `species_given(df, column, value)`, qui renvoie $P(\text{espèce} \mid \text{df[column]} = \text{value})$ : une Series de probabilités, indexée par **toutes** les espèces du DataFrame dans l'ordre alphabétique, avec 0 pour une espèce absente du groupe. Donne-lui une docstring au format NumPy avec au moins les sections `Parameters`, `Returns` et `Examples` (0A.61).

Puis écris au moins **trois** tests pytest qui appellent `species_given`, sur un **petit DataFrame écrit à la main dans chaque test** (un test unitaire ne dépend pas d'un gros fichier de données), et range-les dans la liste `my_tests_23`. Par exemple : les probabilités somment à 1 ; une espèce absente du groupe a une probabilité de 0 ; un cas calculé à la main. Comme en 2.31, la vérification lance vraiment pytest : tes tests doivent **passer** sur une version juste et **échouer** sur chacune des trois versions buggées de la cellule suivante (lis-les). Tes tests peuvent utiliser `species_given`, `pd`, `np`, `math` et `pytest`.

In [ ]:
# The colleague's analysis: P(species | island), one block per island (copy-pasted three times)
biscoe = penguins[penguins["island"] == "Biscoe"]
p_species_biscoe = biscoe["species"].value_counts() / len(biscoe)

dream = penguins[penguins["island"] == "Dream"]
p_species_dream = dream["species"].value_counts() / len(dream)

torgersen = penguins[penguins["island"] == "Torgersen"]
p_species_torgersen = torgersen["species"].value_counts() / len(biscoe)

print(p_species_biscoe.round(3), p_species_dream.round(3), p_species_torgersen.round(3), sep="\n\n")


def species_given_ok(df, column, value):
    """P(species | df[column] == value): a correct version, to test your tests."""
    chosen = df.loc[df[column] == value, "species"]
    return chosen.value_counts(normalize=True).reindex(sorted(df["species"].unique()), fill_value=0.0)


def species_given_bug_1(df, column, value):
    """Bug 1: divides by the size of the whole table instead of the size of the group."""
    chosen = df.loc[df[column] == value, "species"]
    return (chosen.value_counts() / len(df)).reindex(sorted(df["species"].unique()), fill_value=0.0)


def species_given_bug_2(df, column, value):
    """Bug 2: forgets the species that are absent from the group."""
    chosen = df.loc[df[column] == value, "species"]
    return chosen.value_counts(normalize=True).sort_index()


def species_given_bug_3(df, column, value):
    """Bug 3: reverses the condition, P(value | species) instead of P(species | value)."""
    table = pd.crosstab(df["species"], df[column], normalize="index")
    return table[value].reindex(sorted(df["species"].unique()), fill_value=0.0)

In [ ]:
buggy_island_23 = ...   # a) "Biscoe", "Dream" or "Torgersen"


def species_given(df, column, value):
    """TODO: write the NumPy-style docstring here."""
    raise NotImplementedError("species_given() is not written yet")


# Write your test functions here; each one builds a small DataFrame and calls species_given.
# def test_something():
#     df = pd.DataFrame({"species": [...], "island": [...]})
#     assert ...

my_tests_23 = ...   # the list of your test functions

In [ ]:
wb.check("4.23a", buggy_island_23)
with wb.attempt("4.23"):
    result_23 = species_given(penguins, "island", "Dream")
    expected_23 = pd.crosstab(penguins["species"], penguins["island"], normalize="columns")["Dream"].reindex(SPECIES)
    verdict("4.23", isinstance(result_23, pd.Series) and list(result_23.index) == SPECIES
            and np.allclose(result_23.to_numpy(dtype=float), expected_23.to_numpy()),
            "species_given(penguins, \"island\", \"Dream\") est juste (une Series indexée par les trois espèces).",
            "species_given(penguins, \"island\", \"Dream\") doit renvoyer une Series indexée par Adelie, Chinstrap, "
            "Gentoo, égale à la colonne Dream de pd.crosstab(..., normalize=\"columns\").")
    doc_23 = inspect.getdoc(species_given) or ""
    verdict("4.23", all(section in doc_23 for section in ("Parameters", "Returns", "Examples")),
            "la docstring a ses sections Parameters, Returns et Examples.",
            "la docstring doit avoir au moins les sections Parameters, Returns et Examples (format NumPy, 0A.61).")
if not filled(my_tests_23):
    print("⏳ Ex 4.23 : tests pas encore écrits.")
else:
    good_23 = wb.run_pytest(my_tests_23, subject=species_given_ok, name="species_given", preamble="import pandas as pd")
    verdict("4.23", good_23.ok and good_23.passed >= 3, f"tes tests passent sur une version juste ({good_23.passed} cas).",
            "tes tests doivent tous passer sur une version juste, et être au moins 3 (voir le détail au-dessus).")
    if good_23.ok:
        for bug in [species_given_bug_1, species_given_bug_2, species_given_bug_3]:
            result = wb.run_pytest(my_tests_23, subject=bug, name="species_given", preamble="import pandas as pd", quiet=True)
            verdict("4.23", result.failed + result.errors > 0, f"{bug.__name__} est attrapé ({result.failed} test(s) en échec).",
                    f"tes tests passent sur {bug.__name__}, qui est faux : ajoute un test qui le fait échouer.")

## Partie D · 500 hypothèses en log, intervalles de crédibilité, et un défi

Fiche, « au-delà du livre » (1) à (3). Tu écris le calcul du posterior d'un biais en log-probabilités, puis l'intervalle de crédibilité, que tu compares à l'intervalle bootstrap du ch. 2 ; enfin, tu identifies vingt pièces avec le moins de lancers possible.

### Ex 4.24 — coin_bias_posterior : 500 hypothèses en log-probabilités 🔨 ★★★ ⏱️ 35 min
**Objectif :** calculer le posterior du biais d'une pièce sur une grille, en log-probabilités, sans underflow.  
**Prérequis :** Ex 4.16, Ex 4.18 · fiche, au-delà du livre (1) et (2) · **Fil rouge :** synthétique · **mylearn :** `bayes.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/bayes.py`, mêmes règles qu'en 4.14 (NumPy permis, SciPy non). Enregistre, puis relance la cellule de vérification.

Écris `coin_bias_posterior(flips, grid, prior=None)` (lis sa docstring) :
- compte les faces $h$ et les piles $t$ (les lancers ne valent que 0 ou 1, ou `False` et `True` comme ceux de `rng.random(n) < p` ; toute autre valeur lève une `ValueError`) ; les valeurs de la grille sont des biais entre 0 et 1 ; sans prior, prends le prior uniforme sur la grille ;
- calcule en logarithmes $\log P(\theta) + h \log\theta + t \log(1 - \theta)$, puis retranche le maximum, prends l'exponentielle et normalise (4.18) ;
- attention aux extrémités : avec $\theta = 0$, $\log\theta = -\infty$. Si $h = 0$, le terme vaut $\theta^0 = 1$ et ne doit pas donner `nan` ($0 \times (-\infty)$) : n'ajoute $h\log\theta$ que si $h > 0$ (de même pour $t$). `np.errstate(divide="ignore")` fait taire l'avertissement de $\log 0$ ;
- si toutes les hypothèses reçoivent la probabilité 0 (par exemple, une face et une pile avec la grille $\{0, 1\}$), lève une `ValueError`.

Vérifications : 20 000 lancers d'une pièce, sur une grille de 501 biais (`flips_24`, `GRID_24`). La cellule de vérification appelle ta fonction :
a) le MAP (la valeur de la grille de plus grand posterior) ;
b) l'écart-type du posterior, $\sqrt{\sum_\theta (\theta - m)^2\,P(\theta \mid \text{données})}$, où $m$ est la moyenne du posterior ;
puis deux contrôles (l'accord avec `update_discrete` sur 25 lancers, la durée des deux calculs sur 20 000 lancers) et les tests de `coin_bias_posterior`.

Dans tes notes : compare b) à l'erreur typique fréquentiste $\sqrt{\hat\theta(1 - \hat\theta)/n}$ du ch. 2. Pourquoi sont-elles si proches ?

In [ ]:
GRID_24 = np.linspace(0, 1, 501)
flips_24 = wb.synth.coin_flips(20_000, 0.37, seed=24)
print(f"{flips_24.sum()} heads out of {len(flips_24)} flips")

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="4")   # reload your saved file
with wb.attempt("4.24"):
    post_24 = np.asarray(mylearn.bayes.coin_bias_posterior(flips_24, GRID_24), dtype=float)
    wb.check("4.24a", GRID_24[np.argmax(post_24)], computed=True)
    mean_24 = np.sum(GRID_24 * post_24)
    wb.check("4.24b", np.sqrt(np.sum((GRID_24 - mean_24) ** 2 * post_24)), computed=True)
    short_24 = mylearn.bayes.coin_bias_posterior(flips_24[:25], GRID_24)
    loop_24 = mylearn.bayes.update_discrete(np.full(501, 1 / 501), np.column_stack([1 - GRID_24, GRID_24]), flips_24[:25])
    verdict("4.24", np.allclose(short_24, loop_24), "sur 25 lancers, coin_bias_posterior et update_discrete sont d'accord.",
            "sur 25 lancers, coin_bias_posterior doit redonner update_discrete (prior uniforme).")
    with wb.timer("update_discrete, 20 000 flips"):
        mylearn.bayes.update_discrete(np.full(501, 1 / 501), np.column_stack([1 - GRID_24, GRID_24]), flips_24)
    with wb.timer("coin_bias_posterior, 20 000 flips"):
        mylearn.bayes.coin_bias_posterior(flips_24, GRID_24)
    run_bayes_tests("test_coin_bias_posterior_")

### Ex 4.25 — Intervalle de crédibilité contre intervalle bootstrap 🔨 ★★★ ⏱️ 35 min
**Objectif :** calculer un intervalle de crédibilité sur une grille et le comparer à l'intervalle bootstrap du ch. 2, surtout quand les données sont rares.  
**Prérequis :** Ex 4.24, Ex 2.22 (`bootstrap_ci`) · fiche, au-delà du livre (3) · **Fil rouge :** synthétique · **mylearn :** `bayes.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/bayes.py`, mêmes règles qu'en 4.14 (NumPy permis, SciPy non). Enregistre, puis relance la cellule de vérification.

Écris `credible_interval(grid, posterior, mass=0.95)` (lis sa docstring) : cumule le posterior (`np.cumsum`) ; la borne basse est la **première** valeur de la grille dont la probabilité cumulée **atteint** $\frac{1 - \text{mass}}{2}$ (elle est supérieure ou égale), la borne haute la première qui atteint $\frac{1 + \text{mass}}{2}$. Contrôle les entrées (grille strictement croissante, posterior qui est une distribution, `mass` strictement entre 0 et 1) et renvoie un tuple de deux `float` Python.

Vérifications, avec un prior uniforme sur la grille `GRID_25` et ta fonction `coin_bias_posterior` de 4.24. La cellule de vérification appelle tes fonctions :
a) l'intervalle de crédibilité à 95 % des 50 lancers `flips_25` ;
b) l'intervalle de confiance bootstrap des mêmes lancers, avec `mylearn.stats.bootstrap_ci(flips_25, rng=np.random.default_rng(25))` (ta fonction du ch. 2, ou la référence si tu ne l'as pas écrite) ;
c) l'intervalle de crédibilité à 95 % de dix lancers qui ont tous donné pile ;
puis les tests de `credible_interval`.

Enfin, la cellule `coverage_25` répète l'expérience sur 200 petits échantillons (10 lancers d'une pièce de biais 0,1) et compte combien de fois chaque intervalle contient le vrai biais. Dans tes notes : que vaut l'intervalle bootstrap de c) ? Pourquoi ? Quelle méthode tient le mieux sa promesse des 95 % sur les petits échantillons, et pourquoi ?

In [ ]:
GRID_25 = np.linspace(0, 1, 1001)
flips_25 = wb.synth.coin_flips(50, 0.3, seed=25)
TEN_TAILS_25 = np.zeros(10, dtype=int)
print(f"{flips_25.sum()} heads out of {len(flips_25)} flips")


def coverage_25(theta=0.1, n=10, repetitions=200, seed=250):
    """Share of the intervals that contain theta: (credible interval, bootstrap interval), on small samples."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(0, 1, 1001)
    inside_credible = inside_bootstrap = zero_heads = 0
    for _ in range(repetitions):
        flips = (rng.random(n) < theta).astype(int)
        low, high = mylearn.bayes.credible_interval(grid, mylearn.bayes.coin_bias_posterior(flips, grid))
        b_low, b_high = mylearn.stats.bootstrap_ci(flips, rng=rng)
        inside_credible += low <= theta <= high
        inside_bootstrap += b_low <= theta <= b_high
        zero_heads += flips.sum() == 0
    return inside_credible / repetitions, inside_bootstrap / repetitions, zero_heads

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="4")   # reload your saved file
with wb.attempt("4.25"):
    posterior_25 = mylearn.bayes.coin_bias_posterior(flips_25, GRID_25)
    wb.check("4.25a", mylearn.bayes.credible_interval(GRID_25, posterior_25), computed=True)
    wb.check("4.25b", mylearn.stats.bootstrap_ci(flips_25, rng=np.random.default_rng(25)), computed=True)
    tails_25 = mylearn.bayes.credible_interval(GRID_25, mylearn.bayes.coin_bias_posterior(TEN_TAILS_25, GRID_25))
    wb.check("4.25c", tails_25, computed=True)
    print("bootstrap interval of ten tails:", mylearn.stats.bootstrap_ci(TEN_TAILS_25, rng=np.random.default_rng(25)))
    run_bayes_tests("test_credible_interval_")
    credible_share_25, bootstrap_share_25, zero_25 = coverage_25()
    print(f"coverage on 200 samples of 10 flips (bias 0.1): credible {credible_share_25:.2f}, "
          f"bootstrap {bootstrap_share_25:.2f} ({zero_25} samples without any heads)")

### Ex 4.26 — Le détective de pièces : vingt pièces, le moins de lancers possible 🏆 ★★★ ⏱️ 60 min
**Objectif :** identifier le biais de vingt pièces avec un budget de lancers serré, en dépensant les lancers là où l'on est le moins sûr.  
**Prérequis :** Ex 4.24, Ex 4.12 · fiche §4.6, §4.7 · **Fil rouge :** synthétique · **Parcours :** C

**Défi.** L'archéologue t'envoie un sac de vingt pièces. Le biais de chacune est l'une des cinq valeurs de `CANDIDATES_26` (0,2 ; 0,35 ; 0,5 ; 0,65 ; 0,8), mais tu ne sais pas laquelle. Tu peux lancer la pièce $i$ avec `bag.flip(i)`, qui renvoie 1 (face) ou 0 (pile). Chaque lancer compte : tu as droit à **2 400 lancers en tout** pour les vingt pièces (`BUDGET_26`), et tu dois identifier au moins **19 pièces sur 20**.

Écris `detective_26(bag)`, qui renvoie la liste des vingt biais que tu annonces (des valeurs de `CANDIDATES_26`). Commence par la stratégie de base : pour chaque pièce, partir du prior uniforme sur les cinq candidats, lancer et mettre à jour (`update_discrete`) jusqu'à ce que le plus grand posterior dépasse 0,99, puis annoncer le MAP. Sur ce sac, elle dépasse le budget : à toi de faire mieux. La vérification lance ta fonction sur le sac de la graine 434, puis sur 20 autres sacs : le défi est réussi si tu tiens l'objectif sur le sac 434 **et** sur au moins 15 des 20 autres (une bonne stratégie ne dépend pas d'un sac particulier).

Pistes : que coûte un seuil plus bas, en pièces mal identifiées ? Un plafond de lancers par pièce aide-t-il ? Toutes les pièces demandent-elles autant de lancers, et dans quel ordre vaut-il mieux les lancer ? N'ouvre pas `bag._biases` : ce serait tricher.

In [ ]:
CANDIDATES_26 = np.array([0.2, 0.35, 0.5, 0.65, 0.8])    # the possible biases of the coins
BUDGET_26 = 2400                                          # flips allowed for the 20 coins together


class CoinBag:
    """Twenty coins whose biases are hidden (taken from CANDIDATES_26). Flip coin i with bag.flip(i)."""

    def __init__(self, seed):
        self._biases = np.random.default_rng(seed).choice(CANDIDATES_26, size=20)
        self._rngs = [np.random.default_rng([seed, i]) for i in range(20)]   # one stream per coin
        self.flips_used = 0

    def flip(self, i):
        """One flip of coin i: 1 (heads) or 0 (tails). Every call counts in the budget."""
        self.flips_used += 1
        return int(self._rngs[i].random() < self._biases[i])

    def score(self, answers):
        """Number of coins whose bias is right (used by the check: do not peek at self._biases)."""
        return int(np.sum(np.asarray(answers, dtype=float) == self._biases))

In [ ]:
def detective_26(bag):
    """The list of the 20 biases you announce (values of CANDIDATES_26), using bag.flip(i)."""
    raise NotImplementedError("detective_26() is not written yet")

In [ ]:
with wb.attempt("4.26"):
    bag_26 = CoinBag(434)
    answers_26 = detective_26(bag_26)
    score_26, used_26 = bag_26.score(answers_26), bag_26.flips_used
    others_26 = []
    for seed in range(1000, 1020):                       # 20 other bags: the strategy must not depend on one bag
        bag = CoinBag(seed)
        others_26.append((bag.score(detective_26(bag)), bag.flips_used))
    others_26 = np.array(others_26)
    succeeded_26 = int(np.sum((others_26[:, 0] >= 19) & (others_26[:, 1] <= BUDGET_26)))
    print(f"bag 434: {score_26} coins right out of 20, {used_26} flips (budget {BUDGET_26}); "
          f"20 other bags: {succeeded_26} successes, flips used from {others_26[:, 1].min()} to {others_26[:, 1].max()}")
    verdict("4.26", len(answers_26) == 20 and score_26 >= 19 and used_26 <= BUDGET_26 and succeeded_26 >= 15,
            f"défi réussi : {score_26} pièces sur 20 sur le sac 434, en {used_26} lancers, et l'objectif tenu sur "
            f"{succeeded_26} des 20 autres sacs.",
            f"sac 434 : {score_26} pièces sur 20 en {used_26} lancers ; objectif tenu sur {succeeded_26} des 20 autres "
            f"sacs. Il faut au moins 19 pièces sur 20 en {BUDGET_26} lancers au plus sur le sac 434, et l'objectif "
            f"tenu sur au moins 15 des 20 autres sacs.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire la règle de Bayes, nommer ses quatre termes et la retrouver à partir de la règle du produit ?
2. Sais-tu mettre à jour une distribution sur plusieurs hypothèses, observation après observation, sans underflow ?
3. Sais-tu expliquer la différence entre un intervalle de crédibilité et un intervalle de confiance, et quand ils diffèrent ?

**Pour aller plus loin** : la vidéo de 3Blue1Brown et le livre en ligne *Think Bayes*, cités dans la fiche ; l'article de VanderPlas (4.11). La suite : le ch. 5 (courbes et surfaces), puis le ch. 6, qui additionne lui aussi des log-probabilités (entropie, log loss) ; `mylearn.bayes` resservira au ch. 9 (régression « à la Bayes ») et au ch. 13 (Naive Bayes).